In [88]:
%load_ext cudf.pandas

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [89]:
import pickle

# Load the model back
with open('/kaggle/input/datasets/ashura369/my-dataset/df_1.kl', 'rb') as f:
    df = pickle.load(f)

In [90]:
df.head(10)

,id,name,date,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
0,570306133677760513,cairdin,2015-02-24,Tuesday,11,Eastern_Time,Virgin_America,What said.,Flight Booking Problems,0.00000,0,neutral
1,570301130888122368,jnardino,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,plus you've added commercials to the experienc...,Customer Service Issue,0.00000,0,positive
2,570301083672813571,yvonnalynn,2015-02-24,Tuesday,11,Central_Time,Virgin_America,I didn't today... Must mean I need to take ano...,Flight Attendant Complaints,0.00000,0,neutral
3,570301031407624196,jnardino,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,"it's really aggressive to blast obnoxious ""ent...",Bad Flight,0.70330,0,negative
4,570300817074462722,jnardino,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,and it's a really big bad thing about it,Can't Tell,1.00000,0,negative
5,570300767074181121,jnardino,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,seriously would pay $30 a flight for seats tha...,Can't Tell,0.68420,0,negative
6,570300616901320704,cjmcginnis,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,"yes, nearly every time I fly VX this “ear worm...",Customer Service Issue,0.00000,0,positive
7,570300248553349120,pilot,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,Really missed a prime opportunity for Men With...,Customer Service Issue,0.00000,0,neutral
8,570299953286942721,dhepburn,2015-02-24,Tuesday,11,Pacific_Time,Virgin_America,"Well, I didn't…but NOW I DO! :-D",Customer Service Issue,0.00000,0,positive
9,570295459631263746,YupitsTate,2015-02-24,Tuesday,10,Eastern_Time,Virgin_America,"it was amazing, and arrived an hour early. You...",Bad Flight,0.23895,0,positive


In [91]:
df = df.drop(columns=['id','name','date'])

In [92]:
df['reason'] = df['reason'].str.replace(" ", "_")

In [93]:
df.head(5)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
0,Tuesday,11,Eastern_Time,Virgin_America,What said.,Flight_Booking_Problems,0.0000,0,neutral
1,Tuesday,11,Pacific_Time,Virgin_America,plus you've added commercials to the experienc...,Customer_Service_Issue,0.0000,0,positive
2,Tuesday,11,Central_Time,Virgin_America,I didn't today... Must mean I need to take ano...,Flight_Attendant_Complaints,0.0000,0,neutral
3,Tuesday,11,Pacific_Time,Virgin_America,"it's really aggressive to blast obnoxious ""ent...",Bad_Flight,0.7033,0,negative
4,Tuesday,11,Pacific_Time,Virgin_America,and it's a really big bad thing about it,Can't_Tell,1.0000,0,negative


## **Using `MinMaxScaler` on hour, and `StandardScaler` on reason_confidence, and retweets**

In [94]:
data = df.copy()
data.sample(5)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
1637,Sunday,15,Pacific_Time,United,customer service what's That???,Customer_Service_Issue,1.0000,0,negative
7596,Sunday,7,Sydney,Delta,do bags still fly free or have you started cha...,Customer_Service_Issue,0.0000,0,neutral
6920,Monday,16,Eastern_Time,Delta,"we're home, you guys recovered, now we can lau...",Late_Flight,0.6803,0,negative
5619,Friday,11,Quito,Southwest,"4 hour delay in ATL due to ""air traffic contro...",Late_Flight,1.0000,0,negative
14207,Sunday,16,Brasilia,American,24 hours on the AirPort,longlines,0.6900,0,negative


In [95]:
data.select_dtypes(include='object').head()

,day_name,timezone,airlines,feedback,reason,sentiment
0,Tuesday,Eastern_Time,Virgin_America,What said.,Flight_Booking_Problems,neutral
1,Tuesday,Pacific_Time,Virgin_America,plus you've added commercials to the experienc...,Customer_Service_Issue,positive
2,Tuesday,Central_Time,Virgin_America,I didn't today... Must mean I need to take ano...,Flight_Attendant_Complaints,neutral
3,Tuesday,Pacific_Time,Virgin_America,"it's really aggressive to blast obnoxious ""ent...",Bad_Flight,negative
4,Tuesday,Pacific_Time,Virgin_America,and it's a really big bad thing about it,Can't_Tell,negative


In [96]:
data.dtypes[data.dtypes == 'object'].reset_index()

,index,0
0,day_name,object
1,timezone,object
2,airlines,object
3,feedback,object
4,reason,object
5,sentiment,object


In [97]:
data.select_dtypes(include='object').nunique()

day_name         7
timezone        78
airlines         6
feedback     14340
reason          10
sentiment        3
dtype: int64

## Making a function to use tokenization and lemmatization

In [98]:
!pip install cleantext

In [99]:
import nltk
from nltk.tokenize import word_tokenize

from nltk.corpus import stopwords
stop_words = stopwords.words('english')

from nltk.stem import WordNetLemmatizer
lmt = WordNetLemmatizer()

from cleantext import clean

In [100]:
def transform(txt):
    txt = clean(txt, lowercase=True, punct=True)
    txt = word_tokenize(txt)

    temp = [lmt.lemmatize(word, pos='v') for word in txt if word not in stop_words]

    temp2 = temp[:]
    temp2 = " ".join(temp2)

    return temp2

In [101]:
transform('Hello this the God King, playing and singing songs of liberation')

'hello god king play sing songs liberation'

In [102]:
data.select_dtypes(include='object').nunique()

day_name         7
timezone        78
airlines         6
feedback     14340
reason          10
sentiment        3
dtype: int64

In [103]:
data['day_name'] = data['day_name'].apply(transform)
data['timezone'] = data['timezone'].apply(transform)
data['airlines'] = data['airlines'].apply(transform)
data['feedback'] = data['feedback'].apply(transform)
data['reason'] = data['reason'].apply(transform)

In [104]:
data.sample(10)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
40,monday,17,centraltime,virginamerica,view downtown los angeles hollywood sign beyon...,customerserviceissue,0.188742,0,positive
2603,friday,21,easterntime,unite,please call 3107952210,customerserviceissue,0.000000,0,neutral
2112,saturday,21,centraltime,unite,u gate ready gate agent go hour two,lateflight,1.000000,0,negative
2391,saturday,12,pacifictime,unite,hello fly first class behind 20 people zone 1 ...,badflight,0.322100,0,negative
14112,sunday,17,mountaintime,american,pull away gate get departure time leave us tar...,lateflight,0.360200,1,negative
9681,sunday,20,alaska,usairways,worst,canttell,1.000000,0,negative
7074,monday,11,easterntime,delta,lol fleet fleek see yall ballin new jet deck,canttell,0.000000,0,neutral
10468,saturday,15,easterntime,usairways,hello world record attempt amount ball point p...,lateflight,0.000000,0,neutral
3283,thursday,15,centraltime,unite,“ unite happy board please share detail httptc...,lateflight,0.000000,0,positive
7187,monday,9,easterntime,delta,good 2 remind 1914 might horse amp buggy whats...,lateflight,1.000000,0,negative



## Using label encoder on sentiment

In [105]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data[['sentiment']] = le.fit_transform(data[['sentiment']]) 

In [106]:
le.classes_

array(['negative', 'neutral', 'positive'], dtype=object)

In [107]:
data.head(5)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment
0,tuesday,11,easterntime,virginamerica,say,flightbookingproblems,0.0000,0,1
1,tuesday,11,pacifictime,virginamerica,plus youve add commercials experience tacky,customerserviceissue,0.0000,0,2
2,tuesday,11,centraltime,virginamerica,didnt today must mean need take another trip,flightattendantcomplaints,0.0000,0,1
3,tuesday,11,pacifictime,virginamerica,really aggressive blast obnoxious entertainmen...,badflight,0.7033,0,0
4,tuesday,11,pacifictime,virginamerica,really big bad thing,canttell,1.0000,0,0


## Using vectorization to convert all the text into vectors

In [108]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

#### Using ColumnTransformer to add the needed columns to add into one single column

In [109]:
data['combined_labels'] = data[['day_name','timezone','airlines','reason']].astype(str).agg(' '.join, axis=1)
data.sample(6)

,day_name,hour,timezone,airlines,feedback,reason,reason_confidence,retweets,sentiment,combined_labels
3092,friday,0,pacifictime,unite,ill impress actually get response 😜,lateflight,0.0000,0,1,friday pacifictime unite lateflight
11472,wednesday,8,quito,usairways,get home 30 hours suppose live airport offer m...,lateflight,0.6170,0,0,wednesday quito usairways lateflight
82,monday,10,easterntime,virginamerica,youre best whenever begrudgingly use airline i...,lateflight,0.3477,0,0,monday easterntime virginamerica lateflight
2718,friday,17,easterntime,unite,amp top free tv int ’ l leg ’ sit tarmac houst...,lateflight,0.6579,0,0,friday easterntime unite lateflight
6008,thursday,7,easterntime,southwest,chance could get ticket destinationdragons sho...,customerserviceissue,0.0000,0,1,thursday easterntime southwest customerservice...
2882,friday,11,easterntime,unite,book pay flight get denver specific time meet ...,lateflight,0.3763,1,0,friday easterntime unite lateflight


In [110]:
processor = ColumnTransformer(
    transformers=[
        ('labels_tf', TfidfVectorizer(ngram_range=(2,2)), 'combined_labels'),
        ('sentence_tf', TfidfVectorizer(ngram_range=(2,2)), 'feedback'),
        ('minmax', MinMaxScaler(), ['hour']),
        ('standard', StandardScaler(), ['reason_confidence', 'retweets'])
    ],
    remainder='drop'            # will drop rest of the unnecessary columns
)


In [111]:
x = processor.fit_transform(data)
print(len(x.toarray()))
x.toarray()

14640


array([[ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799],
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799],
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799],
       ...,
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799],
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
         0.49054576, -0.11082799],
       [ 0.        ,  0.        ,  0.        , ...,  0.47826087,
        -1.20868402, -0.11082799]])

In [112]:
y = data['sentiment'].values
y

array([1, 2, 1, ..., 1, 0, 1])

# Training the model

In [113]:
from sklearn.model_selection import train_test_split as ttt

x_train, x_test, y_train, y_test = ttt(x, y, test_size=0.3, random_state=42)
print(x_train.shape)
print(x_test.shape)

(10248, 81510)
(4392, 81510)


# Using Optuna to find the best model 

In [118]:
import optuna
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def model(trial):
    classifier_name = trail.suggest_categorical(
        'classifier', [
            'RandomForestClassifier', 'ExtraTreesClassifier', 'GradientBoostingClassifier', 'AdaBoostClassifier', 'BaggingClassifier', 'StackingClassifier'
        ]
    )
    
    # -----------------------------------------------------------------------------------
    # for RandomForest
    # -----------------------------------------------------------------------------------
    if classifier_name == 'RandomForest':
        n_estimators = trial.suggest_int('n_estimators', 50, 500, step=50)
        max_depth = trial.suggest_int('max_depth', 10, 150, step=10)
        criterion = trail.suggest_categorical('criterion', ['gini', 'entropy'])
        min_samples_split = trial.suggest_int('min_samples_split', 2,14, step=2)
        min_samples_leaf = trail.suggest_int('min_samples_leaf', 1, 9, step=2)
        max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])
        class_weight = trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample'])
        
        
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            criterion=criterion,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=bootstrap,
            oob_score=oob_score,
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1 
        )

    # -----------------------------------------------------------------------------------
    # for ExtraTrees
    # -----------------------------------------------------------------------------------

```python

import optuna
import xgboost as xgb
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, 
    AdaBoostClassifier, BaggingClassifier, StackingClassifier
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import warnings

# Ignore warnings for clean output
warnings.filterwarnings('ignore')

def objective(trial):
    # 1. Ask Optuna to choose which model to try
    classifier_name = trial.suggest_categorical(
        "classifier", [
            "RandomForest", "ExtraTrees", "GradientBoosting", "AdaBoost", 
            "XGBoost", "SVC", "Bagging", "Stacking"
        ]
    )
    
    # 2. Define the hyperparameter search spaces based on the chosen model
    if classifier_name == "RandomForest":
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 50, 300, step=50)
        rf_max_depth = trial.suggest_int("rf_max_depth", 10, 100, step=10)
        model = RandomForestClassifier(
            n_estimators=rf_n_estimators, max_depth=rf_max_depth,
            random_state=42, n_jobs=-1
        )
        
    elif classifier_name == "ExtraTrees":
        et_n_estimators = trial.suggest_int("et_n_estimators", 50, 300, step=50)
        et_max_depth = trial.suggest_int("et_max_depth", 10, 100, step=10)
        model = ExtraTreesClassifier(
            n_estimators=et_n_estimators, max_depth=et_max_depth,
            random_state=42, n_jobs=-1
        )
        
    elif classifier_name == "GradientBoosting":
        gb_n_estimators = trial.suggest_int("gb_n_estimators", 50, 200, step=50)
        gb_learning_rate = trial.suggest_float("gb_learning_rate", 0.01, 0.3, log=True)
        model = GradientBoostingClassifier(
            n_estimators=gb_n_estimators, learning_rate=gb_learning_rate,
            random_state=42
        )
        
    elif classifier_name == "AdaBoost":
        ada_n_estimators = trial.suggest_int("ada_n_estimators", 50, 250, step=50)
        ada_learning_rate = trial.suggest_float("ada_learning_rate", 0.01, 1.0, log=True)
        model = AdaBoostClassifier(
            n_estimators=ada_n_estimators, learning_rate=ada_learning_rate,
            random_state=42
        )
        
    elif classifier_name == "XGBoost":
        xgb_n_estimators = trial.suggest_int("xgb_n_estimators", 50, 300, step=50)
        xgb_learning_rate = trial.suggest_float("xgb_learning_rate", 0.01, 0.3, log=True)
        xgb_max_depth = trial.suggest_int("xgb_max_depth", 3, 15)
        model = xgb.XGBClassifier(
            n_estimators=xgb_n_estimators,
            learning_rate=xgb_learning_rate,
            max_depth=xgb_max_depth,
            random_state=42,
            n_jobs=-1,
            eval_metric='mlogloss' # standard for multi-class classification
        )
        
    elif classifier_name == "SVC":
        svc_c = trial.suggest_float("svc_c", 0.1, 10.0, log=True)
        svc_kernel = trial.suggest_categorical("svc_kernel", ["linear", "rbf"])
        model = SVC(
            C=svc_c, kernel=svc_kernel, random_state=42
        )
        
    elif classifier_name == "Bagging":
        bag_n_estimators = trial.suggest_int("bag_n_estimators", 10, 100, step=10)
        bag_max_samples = trial.suggest_float("bag_max_samples", 0.5, 1.0)
        model = BaggingClassifier(
            n_estimators=bag_n_estimators, 
            max_samples=bag_max_samples,
            random_state=42, n_jobs=-1
        )
        
    elif classifier_name == "Stacking":
        # Stacking combines multiple base models. We tune the final meta-model (LogisticRegression)
        stack_final_c = trial.suggest_float("stack_final_c", 0.1, 10.0, log=True)
        
        # Base models for stacking
        base_models = [
            ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
            ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42))
        ]
        
        model = StackingClassifier(
            estimators=base_models,
            final_estimator=LogisticRegression(C=stack_final_c, max_iter=500),
            n_jobs=-1
        )
        
    # 3. Evaluate the chosen model using cross-validation
    # cv=3 means it tests on 3 different chunks of your training data to ensure reliability
    score = cross_val_score(model, x_train, y_train, n_jobs=-1, cv=3, scoring="accuracy")
    
    return score.mean()

# ---------------------------------------------------------
# RUN THE OPTUNA STUDY
# ---------------------------------------------------------
study = optuna.create_study(direction="maximize")

print("Starting MASSIVE Optuna hyperparameter optimization...")
study.optimize(objective, n_trials=40)  # Ran for 40 trials so it has a chance to try everything!

print("\n🎉 Optimization Finished! 🎉")
print(f"Best Accuracy: {study.best_value * 100:.2f}%")
print("Best Model and Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

```